# 🚢 Titanic - Machine Learning from Disaster
**Kaggle Competition | Binary Classification**

---

## Objective
Predict which passengers survived the Titanic shipwreck using passenger data (name, age, gender, socioeconomic class, etc.).

## Dataset
- **train.csv** — 891 rows with survival labels (used to train and validate)
- **test.csv** — 418 rows without labels (used to generate submission)

## Workflow
1. Exploratory Data Analysis (EDA)
2. Feature Engineering
3. Preprocessing & Encoding
4. Model Training & Evaluation
5. Generating Kaggle Submission

> 📌 **Note:** Download the dataset from [Kaggle](https://www.kaggle.com/competitions/titanic/data) and place `train.csv` and `test.csv` in the same directory as this notebook.

---
## 1. Import Libraries

In [ ]:
# Core
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Evaluation
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

# Styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ All libraries imported successfully!')

---
## 2. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape : {test.shape}')
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Missing Values

In [ ]:
def missing_summary(df, name='Dataset'):
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    result = pd.DataFrame({'Missing Count': missing, 'Missing %': pct})
    result = result[result['Missing Count'] > 0].sort_values('Missing %', ascending=False)
    print(f'\n--- {name} ---')
    print(result)

missing_summary(train, 'Train')
missing_summary(test, 'Test')

In [ ]:
# Visualize missing values as a heatmap
plt.figure(figsize=(12, 4))
sns.heatmap(train.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values in Training Data', fontsize=14)
plt.tight_layout()
plt.show()

### 3.2 Survival Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
sns.countplot(data=train, x='Survived', ax=axes[0], palette='Set2')
axes[0].set_title('Survival Count')
axes[0].set_xticklabels(['Did Not Survive (0)', 'Survived (1)'])

# Pie chart
survived_counts = train['Survived'].value_counts()
axes[1].pie(survived_counts, labels=['Did Not Survive', 'Survived'],
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Survival Proportion')

plt.suptitle('Overall Survival Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nSurvival Rate: {train['Survived'].mean():.2%}")

### 3.3 Survival by Key Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# By Sex
sns.barplot(data=train, x='Sex', y='Survived', ax=axes[0], palette='Set1', ci=None)
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)

# By Pclass
sns.barplot(data=train, x='Pclass', y='Survived', ax=axes[1], palette='Set2', ci=None)
axes[1].set_title('Survival Rate by Passenger Class')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)

# By Embarked
sns.barplot(data=train, x='Embarked', y='Survived', ax=axes[2], palette='Set3', ci=None)
axes[2].set_title('Survival Rate by Embarkation Port')
axes[2].set_ylabel('Survival Rate')
axes[2].set_ylim(0, 1)

plt.suptitle('Survival Rate by Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Age & Fare distributions by survival
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
for survived, color, label in zip([0, 1], ['#e74c3c', '#2ecc71'], ['Did Not Survive', 'Survived']):
    train[train['Survived'] == survived]['Age'].dropna().plot.kde(
        ax=axes[0], label=label, color=color, linewidth=2)
axes[0].set_title('Age Distribution by Survival')
axes[0].set_xlabel('Age')
axes[0].legend()

# Fare distribution (log scale for readability)
for survived, color, label in zip([0, 1], ['#e74c3c', '#2ecc71'], ['Did Not Survive', 'Survived']):
    train[train['Survived'] == survived]['Fare'].dropna().plot.kde(
        ax=axes[1], label=label, color=color, linewidth=2)
axes[1].set_title('Fare Distribution by Survival')
axes[1].set_xlabel('Fare')
axes[1].set_xlim(0, 300)
axes[1].legend()

plt.suptitle('Numerical Features by Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Survival heatmap: Sex × Pclass
pivot = train.pivot_table(values='Survived', index='Sex', columns='Pclass', aggfunc='mean')
plt.figure(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn', linewidths=0.5,
            cbar_kws={'label': 'Survival Rate'})
plt.title('Survival Rate: Sex × Passenger Class', fontsize=13)
plt.tight_layout()
plt.show()

**Key Observations from EDA:**
- 🚺 **Women** had a much higher survival rate (~74%) vs men (~19%) — reflecting the "women and children first" protocol
- 💎 **First-class passengers** survived at higher rates than second/third class
- 👶 **Children** (younger ages) had a slightly better survival rate
- 💰 **Higher fare** passengers (correlating with class) showed better survival
- 📍 Passengers who **embarked from Cherbourg (C)** had a higher survival rate (likely due to more 1st class passengers)

The `Cabin` feature has ~77% missing data — we will drop it. `Age` (~20% missing) and `Embarked` (2 rows) will be imputed.

---
## 4. Feature Engineering

Good feature engineering can significantly improve model performance. We'll extract meaningful information from existing columns.

In [ ]:
# Combine train and test for consistent preprocessing
test['Survived'] = np.nan  # placeholder
combined = pd.concat([train, test], ignore_index=True)

print(f'Combined shape: {combined.shape}')

In [ ]:
# ─── Feature: Title (extracted from Name) ───────────────────────────────────
combined['Title'] = combined['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()

# Check title distribution
print('All titles found:')
print(combined['Title'].value_counts())

In [ ]:
# Group rare titles
rare_titles = combined['Title'].value_counts()[combined['Title'].value_counts() < 10].index
combined['Title'] = combined['Title'].replace(rare_titles, 'Rare')
combined['Title'] = combined['Title'].replace({
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'
})

print('\nCleaned titles:')
print(combined['Title'].value_counts())

In [ ]:
# ─── Feature: Family Size ────────────────────────────────────────────────────
combined['FamilySize'] = combined['SibSp'] + combined['Parch'] + 1  # +1 for the passenger

# Categorize family size
combined['FamilyCategory'] = pd.cut(
    combined['FamilySize'],
    bins=[0, 1, 4, 11],
    labels=['Alone', 'Small', 'Large']
)

# ─── Feature: IsAlone ────────────────────────────────────────────────────────
combined['IsAlone'] = (combined['FamilySize'] == 1).astype(int)

# Survival rate by family category (train only)
print('Survival by Family Category:')
print(combined[:len(train)].groupby('FamilyCategory')['Survived'].mean())

In [ ]:
# ─── Feature: HasCabin ───────────────────────────────────────────────────────
# Rather than imputing 77% missing Cabin values, we use presence/absence as a feature
combined['HasCabin'] = combined['Cabin'].notna().astype(int)

# Cabin deck (first letter), fill missing with 'U' for Unknown
combined['Deck'] = combined['Cabin'].str[0].fillna('U')

print('HasCabin survival rate:')
print(combined[:len(train)].groupby('HasCabin')['Survived'].mean())

In [ ]:
# ─── Feature: Fare Binning ───────────────────────────────────────────────────
combined['FareBin'] = pd.qcut(combined['Fare'].fillna(combined['Fare'].median()),
                               q=4, labels=['Low', 'Medium', 'High', 'VeryHigh'])

# ─── Feature: Age Binning ────────────────────────────────────────────────────
# We'll fill Age using median grouped by Title (more accurate than overall median)
age_medians = combined.groupby('Title')['Age'].median()
combined['Age'] = combined.apply(
    lambda row: age_medians[row['Title']] if pd.isna(row['Age']) else row['Age'],
    axis=1
)

combined['AgeBin'] = pd.cut(combined['Age'],
                             bins=[0, 12, 18, 35, 60, 100],
                             labels=['Child', 'Teen', 'Adult', 'MiddleAge', 'Senior'])

print('AgeBin distribution:')
print(combined['AgeBin'].value_counts().sort_index())

In [ ]:
# ─── Visualize new features ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

train_fe = combined[:len(train)]

sns.barplot(data=train_fe, x='Title', y='Survived', ax=axes[0], palette='Set1', ci=None)
axes[0].set_title('Survival by Title')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=train_fe, x='FamilyCategory', y='Survived', ax=axes[1], palette='Set2', ci=None)
axes[1].set_title('Survival by Family Size')
axes[1].set_ylim(0, 1)

sns.barplot(data=train_fe, x='AgeBin', y='Survived', ax=axes[2], palette='Set3', ci=None)
axes[2].set_title('Survival by Age Group')
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Engineered Features vs Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Preprocessing

Encode categorical variables, handle remaining nulls, and prepare the final feature matrix.

In [ ]:
# ─── Handle remaining missing values ─────────────────────────────────────────
# Embarked: 2 missing → fill with mode
combined['Embarked'].fillna(combined['Embarked'].mode()[0], inplace=True)

# Fare: 1 missing in test → fill with median
combined['Fare'].fillna(combined['Fare'].median(), inplace=True)

print('Remaining nulls in key features:')
print(combined[['Age', 'Embarked', 'Fare']].isnull().sum())

In [ ]:
# ─── Encode categorical features ─────────────────────────────────────────────
le = LabelEncoder()

cat_cols = ['Sex', 'Embarked', 'Title', 'FamilyCategory', 'FareBin', 'AgeBin', 'Deck']

for col in cat_cols:
    combined[col + '_Enc'] = le.fit_transform(combined[col].astype(str))

print('✅ Label encoding complete.')

In [ ]:
# ─── Select final features ────────────────────────────────────────────────────
FEATURES = [
    'Pclass',
    'Sex_Enc',
    'Age',
    'Fare',
    'SibSp',
    'Parch',
    'Embarked_Enc',
    'Title_Enc',
    'FamilySize',
    'IsAlone',
    'HasCabin',
    'FamilyCategory_Enc',
    'FareBin_Enc',
    'AgeBin_Enc',
]

TARGET = 'Survived'

# Split back into train/test
train_clean = combined[:len(train)].copy()
test_clean  = combined[len(train):].copy()

X = train_clean[FEATURES]
y = train_clean[TARGET].astype(int)
X_submit = test_clean[FEATURES]

print(f'X shape: {X.shape} | y shape: {y.shape}')
print(f'X_submit shape: {X_submit.shape}')

In [ ]:
# ─── Correlation heatmap of final features ────────────────────────────────────
plt.figure(figsize=(12, 8))
corr = X.assign(Survived=y).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Train/Validation split ───────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (important for Logistic Regression, SVM, KNN)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_submit_sc = scaler.transform(X_submit)

print(f'Train: {X_train.shape} | Validation: {X_val.shape}')

---
## 6. Model Training & Evaluation

We'll train multiple classifiers, compare them with cross-validation, then tune the best one.

In [ ]:
# ─── Baseline model comparison (cross-validation) ────────────────────────────
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM'                 : SVC(probability=True, random_state=42),
    'K-Nearest Neighbors' : KNeighborsClassifier(),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name:<25} | Mean: {scores.mean():.4f} | Std: {scores.std():.4f}')

In [ ]:
# ─── Visualize CV results ─────────────────────────────────────────────────────
cv_df = pd.DataFrame(cv_results)

plt.figure(figsize=(12, 5))
means = cv_df.mean().sort_values(ascending=False)
stds  = cv_df.std()[means.index]

bars = plt.bar(means.index, means.values, yerr=stds.values,
               color=sns.color_palette('Set2', len(means)),
               capsize=5, edgecolor='black', linewidth=0.7)

plt.title('5-Fold Cross-Validation Accuracy (with Std Dev)', fontsize=13)
plt.ylabel('Accuracy')
plt.ylim(0.7, 0.95)
plt.xticks(rotation=15, ha='right')
plt.axhline(means.max(), color='red', linestyle='--', alpha=0.5, label=f'Best: {means.max():.4f}')
plt.legend()
plt.tight_layout()
plt.show()

### 6.1 Hyperparameter Tuning — Random Forest & Gradient Boosting

In [ ]:
# ─── GridSearch: Random Forest ────────────────────────────────────────────────
rf_param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [4, 6, 8, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
rf_grid.fit(X_train_sc, y_train)

print('Best RF Params :', rf_grid.best_params_)
print('Best RF CV Acc :', round(rf_grid.best_score_, 4))

In [ ]:
# ─── GridSearch: Gradient Boosting ───────────────────────────────────────────
gb_param_grid = {
    'n_estimators'  : [100, 200],
    'learning_rate' : [0.05, 0.1, 0.2],
    'max_depth'     : [3, 4, 5],
    'subsample'     : [0.8, 1.0],
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=gb_param_grid,
    cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
gb_grid.fit(X_train_sc, y_train)

print('Best GB Params :', gb_grid.best_params_)
print('Best GB CV Acc :', round(gb_grid.best_score_, 4))

### 6.2 Evaluate Best Model on Validation Set

In [ ]:
# Pick the best model
best_model_name = 'Random Forest' if rf_grid.best_score_ >= gb_grid.best_score_ else 'Gradient Boosting'
best_model = rf_grid.best_estimator_ if best_model_name == 'Random Forest' else gb_grid.best_estimator_

print(f'\n🏆 Best Model: {best_model_name}')

# Validation predictions
y_pred = best_model.predict(X_val_sc)

print(f'Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}')
print()
print('Classification Report:')
print(classification_report(y_val, y_pred, target_names=['Did Not Survive', 'Survived']))

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Not Survived', 'Survived'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title(f'Confusion Matrix — {best_model_name}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Feature Importance ───────────────────────────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=FEATURES)
    importances = importances.sort_values(ascending=True)

    plt.figure(figsize=(9, 6))
    colors = ['#2ecc71' if v > importances.median() else '#3498db' for v in importances]
    importances.plot.barh(color=colors, edgecolor='black', linewidth=0.5)
    plt.title(f'Feature Importances — {best_model_name}', fontsize=13)
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

---
## 7. Generate Kaggle Submission

In [ ]:
# Retrain best model on full training data for the final submission
X_full = scaler.fit_transform(X)   # re-fit scaler on all train data
X_submit_final = scaler.transform(X_submit)

best_model.fit(X_full, y)

# Generate predictions
test_preds = best_model.predict(X_submit_final)

# Build submission DataFrame
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived'   : test_preds
})

submission.to_csv('submission.csv', index=False)
print('✅ submission.csv saved!')
print(f'Survival rate in predictions: {submission.Survived.mean():.2%}')
submission.head(10)

---
## 8. Summary & Next Steps

### What We Did
| Step | Details |
|---|---|
| **EDA** | Explored survival rates across gender, class, port, age, fare |
| **Feature Engineering** | Added Title, FamilySize, IsAlone, HasCabin, AgeBin, FareBin |
| **Preprocessing** | Imputed Age (grouped median), Embarked (mode), encoded categoricals |
| **Modeling** | Compared 6 classifiers via 5-fold cross-validation |
| **Tuning** | GridSearchCV on Random Forest & Gradient Boosting |
| **Submission** | Generated `submission.csv` for Kaggle upload |

### Ideas to Push the Score Further

1. **Try XGBoost / LightGBM** — often outperform sklearn's GBM
2. **Stacking / Voting Ensembles** — combine predictions from multiple models
3. **More feature engineering** — ticket prefix, name length, cabin grouping
4. **Interaction features** — e.g., `Pclass × Sex`, `Age × Pclass`
5. **SHAP values** — for deeper model interpretability
6. **Optuna / Bayesian Optimization** — smarter hyperparameter search

### How to Submit on Kaggle
```
kaggle competitions submit -c titanic -f submission.csv -m "Random Forest tuned"
```
Or upload manually at: https://www.kaggle.com/competitions/titanic/submit